In [ ]:
import os
import pandas as pd
import glob
import torch
from torch.utils import data as torch_data
from torch.nn import functional as torch_functional
from resnet import resnet50
from Dataset import GetLoader2,GetLoaderWithID 
from sklearn.model_selection import StratifiedKFold, KFold
from utils import mkdir, _init_fn, set_seed, load_npy_data_,load_npy_data_with_id, c_index
from train import Trainer
from deepsurvloss import NegativeLogLikelihood, NegativeLogLikelihoodWithRegular
import random
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn

In [ ]:
set_seed(1)
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = 'cuda:0'
train_path = 'datatrain_R'
valid_path = 'datatest_R'
model_path = 'cloud'
model_floder = 'resnet3channels'
model_name = 'train_net_without_kfold_0.pth'
dir_path = os.path.join(model_path, model_floder)
save_path = os.path.join(model_path, model_floder, model_name)
mkdir(dir_path)

In [ ]:
datanp_train, event_train, time_train, ids_train = load_npy_data_with_id(train_path, '1')
datanp_valid, event_valid, time_valid, ids_valid = load_npy_data_with_id(valid_path, '1')

In [ ]:
class MySampler(torch_data.Sampler):
    def __init__(self, data_source, neg_expend):
        self.data_source = data_source
        self.neg_expend = neg_expend
 
    def __iter__(self):
        indices_pos = [] 
        indices_neg = []
        indices = []
        for i in range(len(self.data_source)):
            if self.data_source[i][1] == 1:
                indices_pos.append(i)
            else:
                indices_neg.append(i)
        # random.seed(1)
        random.shuffle(indices_pos)
        random.shuffle(indices_neg)
        
        for i in range(len(indices_pos)):
            indices += [indices_pos[i], ] + indices_neg[i*self.neg_expend: (i+1)*self.neg_expend]
        # print('pos_count:{}, neg_count:{}, indices:{}'.format(len(indices_pos), len(indices_neg), indices))
        return iter(indices)
 
    def __len__(self):
        return len(self.data_source)

In [ ]:
def train_mri_type(mri_type):
    epcho_num = 100 
    patience = 100 
    pos_count = 3
    neg_expend = 3
    
    rst_dfs = []
    
    train_x = datanp_train.transpose(0, 1, 4, 3, 2)
    val_x = datanp_valid.transpose(0, 1, 4, 3, 2)
    train_data_retriever = GetLoaderWithID(train_x, event_train, time_train, ids_train)
    valid_data_retriever = GetLoaderWithID(val_x, event_valid, time_valid, ids_valid)
    train_loader = torch_data.DataLoader(
        train_data_retriever,
        batch_size=pos_count * (neg_expend+1),
        num_workers=0,
        pin_memory=False,
        worker_init_fn=_init_fn,
        # batch_size=41,
        shuffle=False,
        sampler=MySampler(train_data_retriever, neg_expend=neg_expend)
    )
    valid_loader = torch_data.DataLoader(
        valid_data_retriever,
        batch_size=52,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        worker_init_fn=_init_fn,
    )
    model = resnet50()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    criterion = NegativeLogLikelihoodWithRegular(device, model, 0.0001)
    trainer = Trainer(
        model,
        device,
        optimizer,
        # scheduler,  ##
        criterion
    )
    train_surv_list, valid_surv_list, valid_out_df, train_out_df = trainer.fit(
        epcho_num,
        train_loader,
        valid_loader,
        "",
        'ccc',
        patience,
        f'{i}',
        train=True
    )
    return train_surv_list, valid_surv_list, valid_out_df, train_out_df

In [ ]:
results = []
for i in range(5):
    model_name = ''
    modelfiles = None
    if not modelfiles:
        train_surv_list, valid_surv_list, valid_out_df, train_out_df = train_mri_type('T2')
        train_out_df.to_csv(f'ccc/train_{i}.csv')
        valid_out_df.to_csv(f'ccc/valid_{i}.csv')
        max_valid_cindex_x = np.argmax(np.array(valid_surv_list))
        results.append(np.max(np.array(valid_surv_list)))
        plt.cla()
        x=range(len(train_surv_list))
        plt.plot(x, train_surv_list, c='orange', label='train_os_cindex')
        plt.plot(x, valid_surv_list, c='g', label='valid_os_cindex')
        max_valid_cindex_x = np.argmax(np.array(valid_surv_list))
        annolist = []
        for dot in (max_valid_cindex_x, ):
            annolist.append((dot, valid_surv_list[dot]))
            annolist.append((dot, train_surv_list[dot]))
        for anno in annolist:
            plt.annotate(f'{anno[1]:.3f}', xy=anno, xytext=anno)
            plt.scatter(*anno, marker='o', edgecolors='black',c= 'white')
        plt.title('resnet50')
        plt.xlabel('epchos')
        plt.legend()
        plt.savefig(f'ccc/resnet50_surv_{i}.png')

In [ ]:
x=range(len(train_surv_list))
plt.plot(x, train_surv_list, c='orange', label='train_os_cindex')
plt.plot(x, valid_surv_list, c='g', label='valid_os_cindex')
max_valid_cindex_x = np.argmax(np.array(valid_surv_list))
annolist = []
for dot in (max_valid_cindex_x, ):
    annolist.append((dot, valid_surv_list[dot]))
    annolist.append((dot, train_surv_list[dot]))
for anno in annolist:
    plt.annotate(f'{anno[1]:.3f}', xy=anno, xytext=anno)
    plt.scatter(*anno, marker='o', edgecolors='black',c= 'white')
plt.title('resnet50')
plt.xlabel('epchos')
plt.legend()
#plt.savefig('resnet50_surv.png')